In [1]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import gc

@dataclass
class SequenceStore:
    # Per-row arrays
    species_ids: np.ndarray
    gene_ids: np.ndarray
    sequences: list[np.ndarray]
    # Lookup tables
    species_to_gene_to_rows: dict[int, dict[int, list[int]]]
    valid_species: np.ndarray
    # Name lookups
    species_names: list[str]
    gene_names: list[str]
    # Token map used for pre-encoding
    token_to_id: dict[str, int]

    @classmethod
    def from_parquet(
        cls,
        parquet_path: str | Path,
        species_col: str = "species",
        gene_col: str = "gene",
        sequence_col: str = "sequence"
    ) -> "SequenceStore":
        print("[SequenceStore] - Loading Parquet file")
        df = pd.read_parquet(parquet_path, columns=[species_col, gene_col, sequence_col])

        # Basic checks
        print("[SequenceStore] - Checking for missing values")
        if df[species_col].isna().any():
            raise ValueError(f"Column '{species_col}' contains missing values.")
        if df[gene_col].isna().any():
            raise ValueError(f"Column '{gene_col}' contains missing values.")
        if df[sequence_col].isna().any():
            raise ValueError(f"Column '{sequence_col}' contains missing values.")
        
        # Ensure categorical dtype, and remove unused categories to ensure continuous numbering
        print("[SequenceStore] - Ensuring continuous categorical species and gene columns")
        species_cat = df[species_col].astype('category').cat.remove_unused_categories()
        gene_cat = df[gene_col].astype('category').cat.remove_unused_categories()

        # Extract category integer IDs
        print("[SequenceStore] - Extracting category integer IDs")
        species_ids = species_cat.cat.codes.to_numpy(copy=True)
        gene_ids = gene_cat.cat.codes.to_numpy(copy=True)

        # Extract category names
        print("[SequenceStore] - Extracting category names")
        species_names = species_cat.cat.categories.to_list()
        gene_names = gene_cat.cat.categories.to_list()

        #Pre-encode sequences, using tiny vocabulary for DNA / ambiguous bases
        print("[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases ")
        token_to_id = {
            'PAD': 0,
            'A': 1,
            'C': 2,
            'G': 3,
            'T': 4,
            'N': 5,
        }

        encode_array = np.full(256, token_to_id['N'], dtype=np.uint8)
        for token, idx in token_to_id.items():
            if token != 'PAD':
                encode_array[ord(token)] = idx

        def encode_dna(seq):
        # Convert string to ASCII bytes, then use the encode_array as a lookup table
            return encode_array[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)]
        
        sequences = [encode_dna(seq) for seq in df[sequence_col].array]
        
        # Build nested row-index mapping
        print("[SequenceStore] - Building nested row-index mapping")
        species_to_gene_to_rows = defaultdict(lambda: defaultdict(list))
        for row_idx, (sid, gid) in enumerate(zip(species_ids, gene_ids)):
            species_to_gene_to_rows[int(sid)][int(gid)].append(row_idx)

        species_to_gene_to_rows = {
            sid: dict(gene_map)
            for sid, gene_map in species_to_gene_to_rows.items()
        }

        valid_species = np.array(sorted(species_to_gene_to_rows.keys()))

        print("[SequenceStore] - Cleaning up temporary memory")
        del df, species_cat, gene_cat
        gc.collect()

        return cls(
            species_ids=species_ids,
            gene_ids=gene_ids,
            sequences=sequences,
            species_to_gene_to_rows=species_to_gene_to_rows,
            valid_species=valid_species,
            species_names=species_names,
            gene_names=gene_names,
            token_to_id=token_to_id
        )
    
    @property
    def num_rows(self) -> int:
        return len(self.sequences)
    
    @property
    def num_species(self) -> int:
        return len(self.species_names)
    
    @property
    def num_genes(self) -> int:
        return len(self.gene_names)
    
    def summary(self) -> str:
        return (
            f"SequenceStore(num_rows={self.num_rows}, "
            f"num_species={self.num_species}, "
            f"num_genes={self.num_genes}, "
            f"valid_species={len(self.valid_species)})"
        )

In [2]:
path_processed_dataset = Path("output") / "processed_dataset.parquet"
store = SequenceStore.from_parquet(path_processed_dataset)

[SequenceStore] - Loading Parquet file
[SequenceStore] - Checking for missing values
[SequenceStore] - Ensuring continuous categorical species and gene columns
[SequenceStore] - Extracting category integer IDs
[SequenceStore] - Extracting category names
[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases 
[SequenceStore] - Building nested row-index mapping
[SequenceStore] - Cleaning up temporary memory


In [42]:
import torch


@dataclass
class BatchBuilder:
    store: SequenceStore
    species_per_batch: int = 64
    crop_length: int = 512
    rng_seed: int | None = None
    pin_memory: bool = False

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.rng_seed)
        self.pad_id = self.store.token_to_id["PAD"]

        if self.species_per_batch < 1:
            raise ValueError("species_per_batch must be at least 1.")
        if self.crop_length < 1:
            raise ValueError("crop_length must be at least 1.")
        if self.species_per_batch > len(self.store.valid_species):
            raise ValueError(
                "species_per_batch cannot exceed the number of valid species."
            )

    @property
    def batch_size(self) -> int:
        return 2 * self.species_per_batch

    def sample_species(self) -> np.ndarray:
        return self.rng.choice(
            self.store.valid_species,
            size=self.species_per_batch,
            replace=False,
        )

    def sample_two_rows_for_species(self, species_id: int) -> tuple[tuple[int, int], tuple[int, int]]:
        gene_to_rows = self.store.species_to_gene_to_rows[species_id]
        gene_ids = list(gene_to_rows.keys())

        # Case A: species has >= 2 genes
        if len(gene_ids) >= 2:
            gene1, gene2 = self.rng.choice(gene_ids, size=2, replace=False)
            row1 = int(self.rng.choice(gene_to_rows[gene1]))
            row2 = int(self.rng.choice(gene_to_rows[gene2]))
            return (row1, gene1), (row2, gene2)

        # Only one gene available
        gene = gene_ids[0]
        rows = gene_to_rows[gene]

        # Case B: one gene, >= 2 rows
        if len(rows) >= 2:
            row1, row2 = self.rng.choice(rows, size=2, replace=False)
            return (int(row1), gene), (int(row2), gene)

        # Case C: one gene, one row
        row = rows[0]
        return (row, gene), (row, gene)

    def write_crop_into_cpu_tensors(self, seq: np.ndarray, input_ids: torch.Tensor, 
                                    attention_mask: torch.Tensor, batch_idx: int) -> None:
        L = self.crop_length
        seq_len = len(seq)

        if seq_len >= L:
            if seq_len == L:
                crop = seq
            else:
                start = self.rng.integers(0, seq_len - L + 1)
                crop = seq[start:start + L]

            # copy crop into CPU tensor row
            input_ids[batch_idx].copy_(torch.from_numpy(crop.astype(np.int64, copy=False)))
            attention_mask[batch_idx].fill_(True)
            return

        # Shorter than crop length: pad
        input_ids[batch_idx].fill_(self.pad_id)
        attention_mask[batch_idx].fill_(False)

        input_ids[batch_idx, :seq_len].copy_(
            torch.from_numpy(seq.astype(np.int64, copy=False))
        )
        attention_mask[batch_idx, :seq_len] = True

    def sample_batch_cpu(self) -> dict[str, torch.Tensor]:
        sampled_species = self.sample_species()
        B = self.batch_size
        L = self.crop_length

        input_ids = torch.full((B, L), fill_value=self.pad_id, dtype=torch.long, device="cpu")
        attention_mask = torch.zeros((B, L), dtype=torch.bool, device="cpu")
        species_ids = torch.empty(B, dtype=torch.long, device="cpu")
        gene_ids = torch.empty(B, dtype=torch.long, device="cpu")

        batch_idx = 0
        for species_id in sampled_species:
            species_id = int(species_id)
            view1, view2 = self.sample_two_rows_for_species(species_id)

            for row_idx, gene_id in (view1, view2):
                seq = self.store.sequences[row_idx]
                self.write_crop_into_cpu_tensors(
                    seq=seq,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    batch_idx=batch_idx,
                )
                species_ids[batch_idx] = species_id
                gene_ids[batch_idx] = int(gene_id)
                batch_idx += 1

        if self.pin_memory and torch.cuda.is_available():
            input_ids = input_ids.pin_memory()
            attention_mask = attention_mask.pin_memory()
            species_ids = species_ids.pin_memory()
            gene_ids = gene_ids.pin_memory()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "species_ids": species_ids,
            "gene_ids": gene_ids,
        }

    def move_batch_to_device(
        self,
        batch: dict[str, torch.Tensor],
        device: str | torch.device,
        non_blocking: bool = True,
    ) -> dict[str, torch.Tensor]:
        """
        Move a CPU batch to the target device.
        """
        device = torch.device(device)
        return {
            k: v.to(device, non_blocking=non_blocking)
            for k, v in batch.items()
        }

ModuleNotFoundError: No module named 'torch'

In [41]:
builder = BatchBuilder(
    store=store,
    species_per_batch=16,
    crop_length=512,
    rng_seed=None,
)

batch = builder.sample_batch()

for i in range(0, len(batch["species_ids"]), 2):
    sid1 = batch["species_ids"][i].item()
    sid2 = batch["species_ids"][i + 1].item()
    gid1 = batch["gene_ids"][i].item()
    gid2 = batch["gene_ids"][i + 1].item()

    print(
        f"pair {i//2}: species={store.species_names[sid1]}, "
        f"genes=({store.gene_names[gid1]}, {store.gene_names[gid2]})"
    )

pair 0: species=Sufflamen albicaudatum, genes=(COXI, Cytb)
pair 1: species=Neoniphon aurolineatus, genes=(Cytb, COXI)
pair 2: species=Lycodes esmarkii, genes=(ND2, COXI)
pair 3: species=Chlorophthalmus albatrossis, genes=(12S_rRNA, Cytb)
pair 4: species=Myloplus planquettei, genes=(COXII, ND6)
pair 5: species=Malacanthus plumieri, genes=(ND6, ND1)
pair 6: species=Profundulus hildebrandi, genes=(Cytb, ATPase8)
pair 7: species=Lycenchelys tristichodon, genes=(16S_rRNA, Cytb)
pair 8: species=Etropus microstomus, genes=(ND1, ND4)
pair 9: species=Marcusenius cyprinoides, genes=(COXI, Cytb)
pair 10: species=Oligosarcus pintoi, genes=(COXI, ND2)
pair 11: species=Rhynchoconger flavus, genes=(12S_rRNA, 16S_rRNA)
pair 12: species=Brochis ambiacus, genes=(Cytb, ND4)
pair 13: species=Phosichthys argenteus, genes=(COXI, COXI)
pair 14: species=Puntioplites falcifer, genes=(ND4, 12S_rRNA)
pair 15: species=Bolbometopon muricatum, genes=(ND6, Cytb)
